# Validare un modello

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


Hai costruito un modello. Ma quanto è valido?

In questa lezione imparerai a usare la validazione per misurare la qualità del modello. Misurare la qualità è la chiave per migliorare i modelli attraverso iterazioni successive.

# Che cos’è la validazione di un modello

Vorrai valutare quasi tutti i modelli che costruirai. Nella maggior parte delle applicazioni, anche se non in tutte, la misura rilevante della qualità è l’accuratezza predittiva. In altre parole: le previsioni del modello saranno vicine a ciò che accade realmente?

Molte persone commettono un grave errore nel misurare l’accuratezza predittiva: fanno previsioni sui *dati di training* e le confrontano con i valori target degli stessi *dati di training*. Tra poco vedrai il problema di questo approccio e come risolverlo, ma prima pensiamo a come lo applicheremmo.

Dovremmo innanzitutto riassumere la qualità del modello in modo comprensibile. Confrontando i valori previsti e quelli reali di 10.000 case, probabilmente troveresti un misto di previsioni buone e cattive. Scorrere un elenco di 10.000 valori previsti e reali non avrebbe senso. Dobbiamo sintetizzarli in un’unica metrica.

Esistono molte metriche per riassumere la qualità di un modello, ma partiremo da quella chiamata **Mean Absolute Error**, o **MAE**, cioè errore assoluto medio. Analizziamo questa metrica partendo dalla parola «errore».

L’errore di previsione per ogni casa è:<br>
```
error=actual−predicted
```

Quindi, se una casa costa 150.000 dollari e avevi previsto un prezzo di 100.000 dollari, l’errore è di 50.000 dollari.

Con la metrica MAE prendiamo il valore assoluto di ogni errore, trasformando gli errori in valori positivi. Poi calcoliamo la media di questi errori assoluti. Questa è la nostra misura della qualità del modello. In parole semplici, possiamo esprimerla così:

> In media, le nostre previsioni si discostano dal valore reale di circa X.

Per calcolare il MAE ci serve prima un modello. Viene costruito nella cella nascosta qui sotto, che puoi esaminare facendo clic sul pulsante `code`.


In [ ]:
# Codice di caricamento dei dati nascosto.
import pandas as pd

# Carica i dati.
melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
# Escludi le righe con prezzo mancante.
filtered_melbourne_data = melbourne_data.dropna(axis=0)
# Scegli il target e le feature.
y = filtered_melbourne_data.Price
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 
                        'YearBuilt', 'Lattitude', 'Longtitude']
X = filtered_melbourne_data[melbourne_features]

from sklearn.tree import DecisionTreeRegressor
# Definisci il modello.
melbourne_model = DecisionTreeRegressor()
# Addestra il modello.
melbourne_model.fit(X, y)

Una volta ottenuto un modello, ecco come calcoliamo l’errore assoluto medio:


In [ ]:
from sklearn.metrics import mean_absolute_error

predicted_home_prices = melbourne_model.predict(X)
mean_absolute_error(y, predicted_home_prices)

# Il problema dei punteggi «in-sample»

La misura appena calcolata può essere chiamata punteggio «in-sample», cioè sul campione di addestramento. Abbiamo usato un unico «campione» di case sia per costruire il modello sia per valutarlo. Ecco perché è un problema.

Immagina che, nel mercato immobiliare nel suo complesso, il colore della porta non abbia alcuna relazione con il prezzo della casa.

Nel campione di dati usato per costruire il modello, però, tutte le case con porte verdi erano molto costose. Il compito del modello è trovare schemi che consentano di prevedere i prezzi: individuerà quindi questo schema e prevederà sempre prezzi elevati per le case con porte verdi.

Poiché questo schema deriva dai dati di training, il modello sembrerà accurato su quei dati.

Ma se lo schema non si conferma quando il modello incontra nuovi dati, le previsioni saranno molto inaccurate nell’uso pratico.

Il valore pratico dei modelli deriva dalla capacità di fare previsioni su nuovi dati: misuriamo quindi le prestazioni su dati che non sono stati usati per costruire il modello. Il modo più semplice è escludere alcuni dati dalla costruzione del modello e usarli poi per verificare l’accuratezza su dati mai visti prima. Questi dati si chiamano **dati di validazione**.

# Scrivere il codice

La libreria scikit-learn dispone della funzione `train_test_split`, che divide i dati in due parti. Useremo una parte come dati di training per addestrare il modello e l’altra come dati di validazione per calcolare `mean_absolute_error`.

Ecco il codice:


In [ ]:
from sklearn.model_selection import train_test_split

# Dividi feature e target nei set di training e validazione.
# La suddivisione usa un generatore di numeri casuali. Assegnando un numero
# all'argomento random_state si ottiene la stessa suddivisione ogni volta che si
# eseguire nuovamente lo script.
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)
# Definisci il modello.
melbourne_model = DecisionTreeRegressor()
# Addestra il modello.
melbourne_model.fit(train_X, train_y)

# Prevedi i prezzi sui dati di validazione.
val_predictions = melbourne_model.predict(val_X)
print(mean_absolute_error(val_y, val_predictions))

# Wow!

L’errore assoluto medio sui dati di training, «in-sample», era di circa 500 dollari. Sui dati esterni al training, «out-of-sample», supera i 250.000 dollari.

È la differenza tra un modello quasi perfettamente corretto e uno inutilizzabile per la maggior parte degli scopi pratici. Come riferimento, il valore medio delle case nei dati di validazione è di 1,1 milioni di dollari. L’errore sui nuovi dati è quindi circa un quarto del valore medio di una casa.

Ci sono molti modi per migliorare questo modello, per esempio sperimentare per trovare feature migliori o tipi di modello diversi.


# Ora tocca a te
Prima di vedere come migliorare questo modello, prova tu la **[validazione di un modello](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex4.html)**.
